In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# =====================================================
# USTAWIENIA
# =====================================================

ROOT = Path(".").resolve()
print(ROOT)

OUT = ROOT / "figures_chi_comparison"

TRANSITIONS = ["AC", "AB", "BCb"]
FEATURE_SETS = [30, 20, 12]

CALIBRATION = "uncalibrated"

FORMATS = ["pdf"]
DPI = 160


# =====================================================
# OPISY OSI
# =====================================================

X_LABELS = {
    "AC": r"$K_0$",
    "AB": r"$\Delta$",
    "BCb": r"$\Delta$",
}

Y_LABEL = r"$\chi$"


# =====================================================
# KOLORY I MARKERY
# =====================================================

COLORS = [
    "#1f77b4",
    "#d62728",
    "#2ca02c",
    "#ff7f0e",
    "#9467bd",
    "#8c564b",
    "#17becf",
    "#e377c2",
    "#7f7f7f",
    "#bcbd22",
]

MARKERS = [
    "o", "s", "^", "D", "v",
    "P", "X", "*", "<", ">"
]


# =====================================================
# NAZWY MODELI
# =====================================================

MODEL_NAMES_PL = {

    # SUPERVISED
    "Logistic Regression": "Regresja logistyczna",
    "Decision Tree": "Drzewo decyzyjne",
    "Random Forest": "Las losowy",
    "Gradient Boosted Trees": "Wzmocnienie gradientowe",
    "kNN": "kNN",
    "SVM (RBF)": "SVM (RBF)",
    "Neural Network": "Sieć neuronowa",

    # UNSUPERVISED
    "KMeans": "K-średnich",
    "Agglomerative": "Grupowanie aglomeracyjne",
    "Spectral": "Grupowanie spektralne",
    "GaussianMixture": "Mieszanina Gaussowska",
    "MeanShift": "Przesunięcie średniej",
    "Birch": "BIRCH",
    "BayesianGMM": "Bayesowska MG",
    "DBSCAN": "DBSCAN",
}


# =====================================================
# WCZYTYWANIE DANYCH
# =====================================================

def load_transition(root, method, transition):
    """
    Wczytuje dane chi oraz summary dla:

        method:
            SUPERVISED
            UNSUPERVISED

        transition:
            AC
            AB
            BCb
    """

    folder = root / method / transition

    if method == "SUPERVISED":
        prefix = transition

    elif method == "UNSUPERVISED":
        prefix = f"{transition}_UNS"

    else:
        raise ValueError(
            f"Nieznana metoda: {method}"
        )

    chi_path = folder / f"{prefix}_curve_chi.csv"
    summary_path = folder / f"{prefix}_summary.csv"

    chi_df = None
    summary_df = None

    if chi_path.exists():

        chi_df = pd.read_csv(
            chi_path
        )

    else:

        print(
            f"[brak] {chi_path}"
        )

    if summary_path.exists():

        summary_df = pd.read_csv(
            summary_path
        )

    else:

        print(
            f"[brak] {summary_path}"
        )

    return chi_df, summary_df


# =====================================================
# ZAPIS FIGURY
# =====================================================

def save_figure(fig, name):

    OUT.mkdir(
        parents=True,
        exist_ok=True
    )

    for ext in FORMATS:

        path = OUT / f"{name}.{ext}"

        fig.savefig(
            path,
            dpi=DPI,
            bbox_inches="tight",
        )

        print(
            f"  zapisano: {path}"
        )

    plt.close(fig)


# =====================================================
# KOLORY MODELI
# =====================================================

def get_model_colors(dataframes):

    models = []

    for df in dataframes:

        if df is None:
            continue

        if "model" not in df.columns:
            continue

        for model in pd.unique(
            df["model"]
        ):

            if model not in models:

                models.append(
                    model
                )

    return {

        model: COLORS[
            i % len(COLORS)
        ]

        for i, model
        in enumerate(models)
    }


# =====================================================
# MARKERY MODELI
# =====================================================

def get_model_markers(dataframes):

    models = []

    for df in dataframes:

        if df is None:
            continue

        if "model" not in df.columns:
            continue

        for model in pd.unique(
            df["model"]
        ):

            if model not in models:

                models.append(
                    model
                )

    return {

        model: MARKERS[
            i % len(MARKERS)
        ]

        for i, model
        in enumerate(models)
    }


# =====================================================
# FILTROWANIE SUPERVISED
# =====================================================

def filter_supervised_models(df):

    df = df.copy()

    keep = (

        (df["model"] == "Logistic Regression")

        |

        (df["model"] == "kNN")

        |

        (
            ~df["model"].str.startswith(
                "Logistic Regression"
            )

            &

            ~df["model"].str.startswith(
                "kNN"
            )
        )
    )

    return df[keep].copy()


# =====================================================
# RYSOWANIE JEDNEGO PANELU
# =====================================================

def plot_chi_panel(
    ax,
    chi_df,
    transition,
    feature_set,
    model_colors,
    model_markers,
    method,
    linestyle="-",
    prefix=None,
):

    # =================================================
    # OŚ X
    # =================================================

    if transition == "AC":

        x_col = "K0"

    else:

        x_col = "Delta"


    # =================================================
    # FILTROWANIE FEATURE SET
    # =================================================

    df = chi_df[
        chi_df["feature_set"]
        == feature_set
    ].copy()


    # =================================================
    # SUPERVISED
    # =================================================

    if method == "SUPERVISED":

        if "calibration" in df.columns:

            df = df[
                df["calibration"]
                == CALIBRATION
            ].copy()

        df = filter_supervised_models(
            df
        )


    # =================================================
    # UNSUPERVISED
    # =================================================

    elif method == "UNSUPERVISED":

        if "stride" in df.columns:

            min_stride = (

                df
                .groupby("model")["stride"]
                .transform("min")
            )

            df = df[
                df["stride"]
                == min_stride
            ].copy()


    else:

        raise ValueError(
            f"Nieznana metoda: {method}"
        )


    # =================================================
    # BRAK DANYCH
    # =================================================

    if df.empty:

        ax.text(
            0.5,
            0.5,
            "Brak danych",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )

        return


    # =================================================
    # RYSOWANIE KRZYWYCH
    # =================================================

    for model in pd.unique(
        df["model"]
    ):

        group = (

            df[
                df["model"] == model
            ]

            .sort_values(
                x_col
            )

            .copy()
        )


        # ---------------------------------------------
        # CHI
        # ---------------------------------------------

        y = group[
            "chi"
        ].to_numpy()


        # ---------------------------------------------
        # BŁĄD
        # ---------------------------------------------

        yerr = group[
            "chi_err"
        ].to_numpy()


        # ---------------------------------------------
        # WYGLĄD
        # ---------------------------------------------

        color = model_colors[
            model
        ]

        marker = model_markers[
            model
        ]

        label = MODEL_NAMES_PL.get(
            model,
            model,
        )

        if prefix is not None:

            label = (
                f"{prefix} — {label}"
            )


        # ---------------------------------------------
        # WYKRES
        # ---------------------------------------------

        ax.errorbar(

            group[x_col],

            y,

            yerr=yerr,

            fmt=marker,

            linestyle=linestyle,

            markersize=3,

            capsize=2,

            linewidth=1.1,

            color=color,

            label=label,
        )


    # =================================================
    # WYGLĄD
    # =================================================

    ax.grid(
        alpha=0.3,
        linewidth=0.6,
    )

    ax.set_xlabel(
        X_LABELS[
            transition
        ]
    )

    ax.set_ylabel(
        Y_LABEL
    )


    # =================================================
    # AB — OBRÓT ETYKIET
    # =================================================

    if transition == "AB":

        plt.setp(
            ax.get_xticklabels(),
            rotation=45,
            ha="right",
        )


# =====================================================
# GŁÓWNA FIGURA 3 x 3
# =====================================================

def make_chi_figure(
    root,
    group,
    filename,
):

    """
    Układ:

                    30 cech      20 cech      12 cech

        AC          [   ]        [   ]        [   ]

        AB          [   ]        [   ]        [   ]

        BCb         [   ]        [   ]        [   ]
    """

    root = Path(
        root
    )


    # =================================================
    # WCZYTANIE DANYCH
    # =================================================

    datasets = {}

    for transition in TRANSITIONS:

        # ---------------------------------------------
        # SUPERVISED
        # ---------------------------------------------

        sup_chi, sup_summary = (
            load_transition(
                root,
                "SUPERVISED",
                transition,
            )
        )


        # ---------------------------------------------
        # UNSUPERVISED
        # ---------------------------------------------

        uns_chi, uns_summary = (
            load_transition(
                root,
                "UNSUPERVISED",
                transition,
            )
        )


        datasets[transition] = {

            "supervised_chi":
                sup_chi,

            "supervised_summary":
                sup_summary,

            "unsupervised_chi":
                uns_chi,

            "unsupervised_summary":
                uns_summary,
        }


    # =================================================
    # ZBIERANIE DANYCH DO KOLORÓW
    # =================================================

    all_chi = []

    for transition in TRANSITIONS:

        data = datasets[
            transition
        ]

        if group in [
            "SUPERVISED",
            "COMBINED",
        ]:

            if (
                data["supervised_chi"]
                is not None
            ):

                all_chi.append(
                    data[
                        "supervised_chi"
                    ]
                )


        if group in [
            "UNSUPERVISED",
            "COMBINED",
        ]:

            if (
                data["unsupervised_chi"]
                is not None
            ):

                all_chi.append(
                    data[
                        "unsupervised_chi"
                    ]
                )


    if not all_chi:

        print(
            "Brak danych do narysowania."
        )

        return


    # =================================================
    # FILTROWANIE DO LEGENDY
    # =================================================

    filtered_all_chi = []

    for df in all_chi:

        df = df.copy()

        if "calibration" in df.columns:

            df = df[
                df["calibration"]
                == CALIBRATION
            ].copy()

            df = filter_supervised_models(
                df
            )

        filtered_all_chi.append(
            df
        )


    # =================================================
    # KOLORY I MARKERY
    # =================================================

    model_colors = get_model_colors(
        filtered_all_chi
    )

    model_markers = get_model_markers(
        filtered_all_chi
    )


    # =================================================
    # FIGURA 3 x 3
    # =================================================

    fig, axes = plt.subplots(

        nrows=len(
            TRANSITIONS
        ),

        ncols=len(
            FEATURE_SETS
        ),

        figsize=(15, 12),

        sharey=False,
    )


    # =================================================
    # RYSOWANIE PANELI
    # =================================================

    for row, transition in enumerate(
        TRANSITIONS
    ):

        data = datasets[
            transition
        ]

        sup_chi = data[
            "supervised_chi"
        ]

        uns_chi = data[
            "unsupervised_chi"
        ]


        for col, feature_set in enumerate(
            FEATURE_SETS
        ):

            ax = axes[
                row,
                col
            ]


            # =========================================
            # SUPERVISED
            # =========================================

            if group in [
                "SUPERVISED",
                "COMBINED",
            ]:

                if sup_chi is not None:

                    plot_chi_panel(

                        ax=ax,

                        chi_df=sup_chi,

                        transition=transition,

                        feature_set=feature_set,

                        model_colors=model_colors,

                        model_markers=model_markers,

                        method="SUPERVISED",

                        linestyle="-",

                        prefix=(
                            "S"
                            if group == "COMBINED"
                            else None
                        ),
                    )


            # =========================================
            # UNSUPERVISED
            # =========================================

            if group in [
                "UNSUPERVISED",
                "COMBINED",
            ]:

                if uns_chi is not None:

                    plot_chi_panel(

                        ax=ax,

                        chi_df=uns_chi,

                        transition=transition,

                        feature_set=feature_set,

                        model_colors=model_colors,

                        model_markers=model_markers,

                        method="UNSUPERVISED",

                        linestyle="--",

                        prefix=(
                            "U"
                            if group == "COMBINED"
                            else None
                        ),
                    )


    # =================================================
    # NAGŁÓWKI KOLUMN
    # =================================================

    for col, feature_set in enumerate(
        FEATURE_SETS
    ):

        axes[0, col].set_title(

            f"{feature_set} cech",

            fontsize=13,

            fontweight="bold",

            pad=12,
        )


    # =================================================
    # OPISY WIERSZY
    # =================================================

    for row, transition in enumerate(
        TRANSITIONS
    ):

        axes[row, 0].annotate(

            transition,

            xy=(
                0,
                0.5
            ),

            xytext=(
                -0.15,
                0.5
            ),

            xycoords="axes fraction",

            textcoords="axes fraction",

            rotation=90,

            va="center",

            ha="center",

            fontsize=13,

            fontweight="bold",
        )


    # =================================================
    # WSPÓLNA LEGENDA
    # =================================================

    handles = []
    labels = []


    # MODELE

    for model in model_colors:

        handles.append(

            Line2D(

                [],

                [],

                color=model_colors[
                    model
                ],

                marker=model_markers[
                    model
                ],

                linestyle="-",

                markersize=5,

                linewidth=1.2,
            )
        )

        labels.append(
            MODEL_NAMES_PL.get(
                model,
                model,
            )
        )


    # SUPERVISED / UNSUPERVISED

    if group == "COMBINED":

        handles.extend(

            [

                Line2D(
                    [],
                    [],
                    color="black",
                    linestyle="-",
                    linewidth=1.5,
                ),

                Line2D(
                    [],
                    [],
                    color="black",
                    linestyle="--",
                    linewidth=1.5,
                ),
            ]
        )

        labels.extend(

            [

                "modele nadzorowane",

                "modele nienadzorowane",
            ]
        )


    # =================================================
    # LEGENDA
    # =================================================

    fig.legend(

        handles,

        labels,

        loc="lower center",

        bbox_to_anchor=(
            0.5,
            -0.005
        ),

        ncol=4,

        fontsize=8,

        frameon=False,
    )


    # =================================================
    # UKŁAD
    # =================================================

    plt.tight_layout(

        rect=[

            0.05,

            0.09,

            1.00,

            0.96,
        ]
    )


    # =================================================
    # ZAPIS
    # =================================================

    save_figure(

        fig,

        filename,
    )


# =====================================================
# GENEROWANIE FIGUR
# =====================================================

make_chi_figure(

    ROOT,

    "SUPERVISED",

    "chi_supervised",
)


make_chi_figure(

    ROOT,

    "UNSUPERVISED",

    "chi_unsupervised",
)


make_chi_figure(

    ROOT,

    "COMBINED",

    "chi_combined",
)

/home/mariuszoslaw/uni/masters/Results
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_chi_comparison/chi_supervised.pdf
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_chi_comparison/chi_unsupervised.pdf
  zapisano: /home/mariuszoslaw/uni/masters/Results/figures_chi_comparison/chi_combined.pdf
